# 原生 Notebook 富输出验收 😀

这是未执行夹具。所有 Python 单元的输出和执行次数均为空。

**原生 Markdown**、*强调*、`行内代码` 与表格：

| 项目 | 目标 |
|---|---|
| PNG/JPEG | 原生图像 |
| HTML/DataFrame | 原生表格 |
| display_id | 显示更新后的图与 99 |

先选择指定 Python 3.14 环境并使用普通项目 Trust 流程。
`Run All` 最后进入 120 秒的可中断单元；点击 **Interrupt**，再用
**Shift+Enter** 运行最后的恢复单元（预期 56）。
此文本只描述待验证步骤，不是运行证据。


In [ ]:
# 1. 真实 Python / producer imports — no installation or network request
import os, sys, time, io
from pathlib import Path
os.environ['MPLCONFIGDIR'] = str(Path.cwd() / '.rich-probe-cache' / 'matplotlib')
os.environ['XDG_CACHE_HOME'] = str(Path.cwd() / '.rich-probe-cache' / 'cache')
import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image as PillowImage, ImageDraw
from IPython.display import display, update_display, Image, HTML, Markdown
print('真实原生富输出 ✅ 中文 + emoji 😀', sys.version.split()[0])
print('pandas', pd.__version__, 'matplotlib', matplotlib.__version__)


In [ ]:
# 2. 实际 Matplotlib PNG；同 display_id 从 phase 1 更新到 phase 2
def native_chart(values, title):
    figure, axes = plt.subplots(figsize=(4, 2.6), dpi=100)
    axes.plot([1, 2, 3], values, marker='o', color='#2783bb')
    axes.set_title(title)
    axes.set_xlabel('sample')
    axes.grid(alpha=0.2)
    data = io.BytesIO()
    figure.savefig(data, format='png', bbox_inches='tight')
    plt.close(figure)
    return Image(data=data.getvalue(), format='png')
display(native_chart([1, 4, 2], 'Native PNG: phase 1'), display_id='native-ui-chart',
        metadata={'image/png': {'width': 420}, 'foreign': {'phase': 1, 'large': 9007199254740993}})
time.sleep(0.6)
update_display(native_chart([3, 1, 5], 'Native PNG: phase 2'), display_id='native-ui-chart',
               metadata={'image/png': {'width': 420}, 'foreign': {'phase': 2, 'large': 9007199254740993}})
print('图像 display_id 更新请求已完成；应只保留 phase 2 图 ✅')


In [ ]:
# 3. 实际 Pillow JPEG
jpeg_image = PillowImage.new('RGB', (300, 110), '#236aa4')
drawing = ImageDraw.Draw(jpeg_image)
drawing.rectangle((12, 12, 287, 97), outline='white', width=3)
drawing.text((38, 44), 'Native JPEG - actual kernel', fill='white')
jpeg_bytes = io.BytesIO()
jpeg_image.save(jpeg_bytes, format='JPEG')
display(Image(data=jpeg_bytes.getvalue(), format='jpeg'),
        metadata={'image/jpeg': {'width': 300}, 'foreignJPEG': {'keep': True}})


In [ ]:
# 4. 实际 pandas HTML；同 display_id 将第一行分数 42 更新到 99
native_frame = pd.DataFrame({'城市 😀': ['北京', '上海'], '分数': [42, 88]})
display(native_frame, display_id='native-ui-frame', metadata={'foreign': {'phase': 1}})
time.sleep(0.6)
native_frame.loc[0, '分数'] = 99
update_display(native_frame, display_id='native-ui-frame', metadata={'foreign': {'phase': 2}})
print('表格 display_id 更新请求已完成；北京应显示 99 ✅')


In [ ]:
# 5. 实际 HTML 和 Markdown 输出，均为静态内容
display(HTML('<h3>静态 HTML ✅</h3><table><tr><th>项目</th><th>值</th></tr>'
             '<tr><td>中文 emoji 😀</td><td>原生 JTable</td></tr></table>'),
        metadata={'foreignHTML': '保留'})
display(Markdown('# 内核 Markdown 😀\n\n**加粗**、*强调* 与 `code`\n\n'
                 '| 项目 | 结果 |\n|---|---|\n| Unicode | 中文 ✅ |'))
display({'text/plain': '未知 MIME 仍应保留', 'application/x-native-rich-evidence+json':
         {'large': 9007199254740993, 'keep': ['中文', '😀']}}, raw=True,
        metadata={'foreignOutput': {'keep': True}})


In [ ]:
# 6. 已知预期错误：需要显示 ValueError traceback，内核本身仍可继续运行
raise ValueError('预期的单元错误：中文 + emoji 😀')


In [ ]:
# 7. 前一单元 ValueError 后，Run All 应继续；这里实际结果应为 42
print('错误后的内核仍可用 ✅', 6 * 7)


In [ ]:
# 8. 中断控制：看到提示后点击工具栏 Interrupt；最长 120 秒，不启动其他进程
import time
print('请现在点击 Interrupt；等待中的真实内核单元 ⏳', flush=True)
time.sleep(120)
print('本次未中断，120 秒自然结束')


In [ ]:
# 9. Interrupt 清空 Run All 队列后，请在本单元按 Shift+Enter；实际结果应为 56
print('中断后恢复 + Shift+Enter ✅ 😀', 7 * 8)
print('当前内核变量仍在：', native_frame.loc[0, '分数'])
